# 03: Blocking Deviation Analysis

**Thesis:** Impact of Machine-Learning Traffic Classification Errors on Multi-Class Capacity Dimensioning in Multirate Loss Systems

This notebook applies the composition $\hat{\mathbf{a}} = \mathbf{C}^\top \mathbf{a}$ to all confusion matrices
from notebook 02 and computes blocking deviations, sensitivity tensors, capacity overhead curves,
and minimum recall thresholds.

Classification errors cause harm in both directions:
- **Low-to-high ($t_j > t_i$):** Misclassification inflates apparent demand, requiring extra capacity (system cost).
- **High-to-low ($t_j < t_i$):** Misclassification deflates apparent demand, causing QoS degradation per flow (user cost).

Every section reports both the capacity overhead $\Delta V / V$ and the weighted bandwidth deficit (WBD).

> The published figures are rendered by the modules in `src/figures/` through
> `scripts/regenerate_all_figures.py`, and the distributed archive is written by
> `scripts/regenerate_analytical_results.py`. The plots here are working views and are not saved.

## 1. Setup and Imports

In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore', category=FutureWarning)

# Path setup
sys.path.insert(0, os.path.abspath('..'))
from src.analytical.kaufman_roberts import (
    kaufman_roberts, bridge_equation, blocking_deviation,
    capacity_overhead, sensitivity_analysis, minimum_recall_search,
)

%matplotlib inline
sns.set_style('whitegrid')
# Working-view rcParams for this notebook; the published figures are rendered by src/figures/style.py at its own dpi and font sizes.
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
})

print("Imports successful.")

In [ ]:
# Load confusion matrices from Notebook 02
data = np.load('../data/processed/confusion_matrices.npz', allow_pickle=True)
CLASS_ORDER = list(data['class_order'])  # ['Browsing', 'Chat', 'FileTransfer', 'Streaming', 'VoIP']
T_K = data['t_k']                       # [2, 1, 8, 15, 1]

# The 9 empirical CMs (3 FlowPic published + 6 trained in notebook 02)
EMPIRICAL_CM_NAMES = [
    'flowpic_nonvpn', 'flowpic_vpn', 'flowpic_tor',
    'xgb_clean', 'mlp_clean',
    'xgb_vpn_shift', 'mlp_vpn_shift',
    'xgb_reduced_feat', 'mlp_reduced_feat',
]

# Human-readable labels for plots
CM_LABELS = {
    'flowpic_nonvpn': 'FlowPic NonVPN',
    'flowpic_vpn': 'FlowPic VPN',
    'flowpic_tor': 'FlowPic Tor',
    'xgb_clean': 'XGB Clean',
    'mlp_clean': 'MLP Clean',
    'xgb_vpn_shift': 'XGB VPN-shift',
    'mlp_vpn_shift': 'MLP VPN-shift',
    'xgb_reduced_feat': 'XGB Reduced',
    'mlp_reduced_feat': 'MLP Reduced',
}

# Load all empirical CMs into a dict
empirical_cms = {}
for name in EMPIRICAL_CM_NAMES:
    empirical_cms[name] = data[name].copy()

print(f"Classes: {CLASS_ORDER}")
print(f"Demands t_k: {T_K}")
print(f"Loaded {len(empirical_cms)} empirical CMs: {list(empirical_cms.keys())}")

In [ ]:
# Scenario parameters: the frozen constants of src/analytical/constants.py
from src.analytical.constants import A_OTT, T_OTT, A_5G, T_5G, CLASS_ORDER_5G, B_TARGET_DEFAULT

a_OTT, t_OTT = A_OTT, T_OTT      # OTT/IPTV, 5 classes: Browsing, Chat, FT, Streaming, VoIP
a_5G, t_5G = A_5G, T_5G          # 5G slicing, 3 classes: eMBB, mMTC, URLLC
CLASSES_5G = list(CLASS_ORDER_5G)
B_TARGET = B_TARGET_DEFAULT      # 1% grade of service

# Nominal capacities at the blocking target
V_OTT = capacity_overhead(a_OTT, t_OTT, B_TARGET, V_start=1)
V_5G = capacity_overhead(a_5G, t_5G, B_TARGET, V_start=1)

# Working palette for this notebook (matplotlib tab10); the thesis figures use the Okabe-Ito palette in src/figures/style.py
CLASS_COLORS = {
    'Browsing':     '#1f77b4',
    'Chat':         '#ff7f0e',
    'FileTransfer': '#2ca02c',
    'Streaming':    '#d62728',
    'VoIP':         '#9467bd',
}
CLASS_COLORS_5G = {
    'eMBB':  '#1f77b4',
    'mMTC':  '#ff7f0e',
    'URLLC': '#d62728',
}

print("OTT/IPTV scenario:")
print(f"  V={V_OTT}, a={a_OTT}, t={t_OTT}")
print(f"  Total offered AU·Erl: {np.sum(a_OTT * t_OTT):.1f}")
print(f"\n5G scenario:")
print(f"  V={V_5G}, a={a_5G}, t={t_5G}")
print(f"  Total offered AU·Erl: {np.sum(a_5G * t_5G):.1f}")

In [ ]:
# Helper functions; the framework functions come from src/analytical

from src.analytical.kaufman_roberts import fix_zero_rows
from src.analytical.scenarios import compute_wbd, make_5g_cm
from src.analytical.recall_thresholds import per_class_recall_search

def compute_accuracy(C):
    """Balanced accuracy: mean of per-class recalls (diagonal entries). Skips zero rows (absent classes)."""
    diag = np.diag(C)
    row_sums = C.sum(axis=1)
    active = row_sums > 1e-10
    if active.sum() == 0:
        return 0.0
    return np.mean(diag[active])

def dominant_direction(C, a, t):
    """Determine whether low-to-high or high-to-low misclassification dominates.
    
    Returns 'low-to-high' if total apparent demand increases, 'high-to-low' otherwise.
    """
    a_hat = bridge_equation(fix_zero_rows(C), a)
    true_demand = np.sum(a * t)
    apparent_demand = np.sum(a_hat * t)
    if apparent_demand > true_demand:
        return 'low-to-high'
    elif apparent_demand < true_demand:
        return 'high-to-low'
    else:
        return 'balanced'

def plot_sensitivity(S, title):
    """Five per-class sensitivity heatmaps plus a shared colourbar."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(title, fontsize=16)
    vmax = np.max(np.abs(S))
    for k in range(5):
        ax = axes[k // 3, k % 3]
        im = ax.imshow(S[k], cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='equal')
        ax.set_title(f'$B_{{{CLASS_ORDER[k]}}}$', fontsize=13, pad=8)
        ax.set_xticks(range(5))
        ax.set_xticklabels(CLASS_ORDER, rotation=45, ha='right', fontsize=9)
        ax.set_yticks(range(5))
        ax.set_yticklabels(CLASS_ORDER, fontsize=9)
        ax.set_ylabel('True class $i$', fontsize=10)
        ax.set_xlabel('Predicted class $j$', fontsize=10)
        for i in range(5):
            for j in range(5):
                val = S[k, i, j]
                color = 'white' if abs(val) > vmax * 0.6 else 'black'
                ax.text(j, i, f'{val:.4f}', ha='center', va='center', fontsize=8, color=color)
    axes[1, 2].set_visible(False)
    cbar_ax = fig.add_axes([0.72, 0.08, 0.02, 0.35])
    fig.colorbar(im, cax=cbar_ax, label='$\\partial B_k / \\partial C_{ij}$')
    plt.subplots_adjust(hspace=0.4, wspace=0.3, top=0.93)
    plt.show()

def print_top_sensitivities(S, title, n=10):
    """The n largest off-diagonal |S[k, i, j]| entries, with their AU demands."""
    entries = sorted(
        (abs(S[k, i, j]), S[k, i, j], k, i, j)
        for k in range(5) for i in range(5) for j in range(5) if i != j
    )
    entries.reverse()
    print(title)
    print(f"{'Rank':>4s}  {'B_k':>12s}  {'True(i)':>12s}  {'Pred(j)':>12s}  {'t_i':>4s}  {'t_j':>4s}  {'S[k,i,j]':>10s}")
    for rank, (_, val, k, i, j) in enumerate(entries[:n], 1):
        print(f"{rank:4d}  {CLASS_ORDER[k]:>12s}  {CLASS_ORDER[i]:>12s}  {CLASS_ORDER[j]:>12s}"
              f"  {t_OTT[i]:4d}  {t_OTT[j]:4d}  {val:+10.6f}")

def overhead_wbd_sweep(cm_for, recalls, a, t, V_nom):
    """Capacity overhead (percent) and WBD across a recall grid for one spillover model."""
    oh, wbd = [], []
    for r in recalls:
        C = fix_zero_rows(cm_for(r))
        a_hat = bridge_equation(C, a)
        try:
            V_prime = capacity_overhead(a_hat, t, B_TARGET, V_start=V_nom, V_max=2000)
            oh.append((V_prime - V_nom) / V_nom * 100)
        except ValueError:
            oh.append(float('nan'))
        wbd.append(compute_wbd(C, a, t))
    return oh, wbd

def rstar_by_eps(a, t, V_nom, epsilons):
    """Per-class r*_k for each epsilon, keyed by epsilon."""
    return {eps: per_class_recall_search(a, t, V_nom, B_TARGET, eps) for eps in epsilons}

print("Helper functions defined.")


## 2. Baseline Blocking (No Classification Errors)

With a perfect classifier ($\mathbf{C} = \mathbf{I}$), the K-R recursion gives the true per-class blocking probabilities $B_k(\mathbf{a}, V)$.


In [ ]:
# Baseline blocking: OTT/IPTV scenario
P_ott, B_ott_baseline = kaufman_roberts(V_OTT, a_OTT, t_OTT)

print(f"OTT/IPTV baseline blocking (perfect classifier, V={V_OTT}):")
print("=" * 50)
for i, cls in enumerate(CLASS_ORDER):
    print(f"  {cls:14s}  t_k={t_OTT[i]:2d}  a_k={a_OTT[i]:5.1f}  B_k={B_ott_baseline[i]:.6f} ({B_ott_baseline[i]*100:.3f}%)")

print(f"\n  Max blocking: {B_ott_baseline.max():.6f} ({B_ott_baseline.max()*100:.3f}%)")
print(f"  Total offered AU·Erl: {np.sum(a_OTT * t_OTT):.1f}")
print(f"  Capacity utilization: {np.sum(a_OTT * t_OTT) / V_OTT * 100:.1f}%")

In [ ]:
# Baseline blocking: 5G scenario
P_5g, B_5g_baseline = kaufman_roberts(V_5G, a_5G, t_5G)

print(f"5G slicing baseline blocking (perfect classifier, V={V_5G}):")
print("=" * 50)
for i, cls in enumerate(CLASSES_5G):
    print(f"  {cls:8s}  t_k={t_5G[i]:2d}  a_k={a_5G[i]:5.1f}  B_k={B_5g_baseline[i]:.6f} ({B_5g_baseline[i]*100:.3f}%)")

print(f"\n  Max blocking: {B_5g_baseline.max():.6f} ({B_5g_baseline.max()*100:.3f}%)")
print(f"  Total offered AU·Erl: {np.sum(a_5G * t_5G):.1f}")
print(f"  Capacity utilization: {np.sum(a_5G * t_5G) / V_5G * 100:.1f}%")

## 3. Composition: True vs Apparent Loads

The composition $\hat{\mathbf{a}} = \mathbf{C}^\top \mathbf{a}$ maps true offered loads through the confusion matrix to produce the apparent (distorted) loads that the capacity dimensioning system would use. The total AU·Erl demand $\sum_k \hat{a}_k t_k$ compared to $\sum_k a_k t_k$ reveals whether low-to-high or high-to-low misclassification dominates.

In [ ]:
# Bridge equation analysis for all 9 empirical CMs
true_demand_total = np.sum(a_OTT * t_OTT)

bridge_rows = []
print("Composition: True vs Apparent Loads")

for name in EMPIRICAL_CM_NAMES:
    C = fix_zero_rows(empirical_cms[name])
    a_hat = bridge_equation(C, a_OTT)
    apparent_demand = np.sum(a_hat * t_OTT)
    pct_change = (apparent_demand - true_demand_total) / true_demand_total * 100
    direction = 'UP' if apparent_demand > true_demand_total else 'DOWN'
    
    print(f"\n{CM_LABELS[name]}:")
    print(f"  {'Class':14s} {'a_true':>8s} {'a_hat':>8s} {'diff':>8s}")
    for i, cls in enumerate(CLASS_ORDER):
        diff = a_hat[i] - a_OTT[i]
        print(f"  {cls:14s} {a_OTT[i]:8.2f} {a_hat[i]:8.2f} {diff:+8.2f}")
    print(f"  AU·Erl:  {true_demand_total:8.1f} {apparent_demand:8.1f}  ({pct_change:+.2f}% {direction})")
    
    bridge_rows.append({
        'CM': CM_LABELS[name],
        'sum(a*t) true': true_demand_total,
        'sum(a_hat*t)': apparent_demand,
        'Direction': direction,
        '% Change': pct_change,
    })

bridge_df = pd.DataFrame(bridge_rows)
print("\n\nSummary Table:")
print(bridge_df.to_string(index=False, float_format='%.2f'))

## 4. Blocking Deviation Analysis

$\Delta B_k = B_k(\hat{\mathbf{a}}, V) - B_k(\mathbf{a}, V)$ quantifies the per-class blocking probability shift induced by classification errors. The capacity overhead $\Delta V / V$ measures system cost; the WBD measures user cost.

In [ ]:
# Blocking deviation + dual-impact metrics for all 9 CMs

# First, find V_nominal: the capacity where max B_k(a, V) just meets B_TARGET for true loads. This is the capacity a perfect classifier would require.
V_nominal = capacity_overhead(a_OTT, t_OTT, B_TARGET, V_start=1, V_max=500)
print(f"Nominal capacity V* (for B_target={B_TARGET}): {V_nominal} AUs")
_, B_at_nominal = kaufman_roberts(V_nominal, a_OTT, t_OTT)
print(f"Blocking at V*: {[f'{b:.6f}' for b in B_at_nominal]}")
print(f"Max blocking at V*: {B_at_nominal.max():.6f}\n")

master_rows = []
all_delta_B = {}

for name in EMPIRICAL_CM_NAMES:
    C = fix_zero_rows(empirical_cms[name])
    result = blocking_deviation(V_nominal, a_OTT, C, t_OTT)
    
    # Capacity overhead: find V' such that B_k(a_hat, V') <= B_TARGET for all k
    a_hat = result['a_hat']
    try:
        V_prime = capacity_overhead(a_hat, t_OTT, B_TARGET, V_start=V_nominal, V_max=2000)
        overhead_pct = (V_prime - V_nominal) / V_nominal * 100
    except ValueError:
        V_prime = None
        overhead_pct = float('inf')
    
    wbd = compute_wbd(C, a_OTT, t_OTT)
    acc = compute_accuracy(C)
    direction = dominant_direction(C, a_OTT, t_OTT)
    
    all_delta_B[name] = result['delta_B']
    
    master_rows.append({
        'CM': CM_LABELS[name],
        'Accuracy': acc,
        'Max |Delta_B|': np.max(np.abs(result['delta_B'])),
        'V_prime': V_prime,
        'Overhead %': overhead_pct,
        'WBD (AU·Erl)': wbd,
        'Direction': direction,
    })

master_df = pd.DataFrame(master_rows)
print("\nMaster Table: Blocking Deviation Analysis")
print(master_df.to_string(index=False, float_format='%.4f'))

In [ ]:
# Grouped bar chart: Delta_B_k per class for all 9 CMs
fig, ax = plt.subplots(figsize=(16, 7))

n_cms = len(EMPIRICAL_CM_NAMES)
n_classes = len(CLASS_ORDER)
x = np.arange(n_classes)
width = 0.8 / n_cms
colors = plt.cm.tab10(np.linspace(0, 1, n_cms))

for idx, name in enumerate(EMPIRICAL_CM_NAMES):
    delta_b = all_delta_B[name]
    offset = (idx - n_cms / 2 + 0.5) * width
    bars = ax.bar(x + offset, delta_b * 100, width, label=CM_LABELS[name],
                  color=colors[idx], edgecolor='white', linewidth=0.5)

ax.set_xlabel('Traffic Class', fontsize=12)
ax.set_ylabel('$\\Delta B_k$ (percentage points)', fontsize=12)
ax.set_title('Per-Class Blocking Deviation Across All Empirical Classifiers', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_ORDER, fontsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()


## 5. Sensitivity Heatmaps

The sensitivity tensor $S[k, i, j] = \partial B_k / \partial C_{ij}$ quantifies how each confusion matrix entry affects each class's blocking probability. Via the chain rule through the composition:

$$S[k, i, j] = \frac{\partial B_k}{\partial \hat{a}_j} \cdot a_i$$

Entries where $a_i$ is large AND $t_j \gg t_i$ (low-to-high misclassification of a high-volume class) dominate the sensitivity landscape.

In [ ]:
# Sensitivity heatmaps: XGBoost Clean CM
C_xgb = fix_zero_rows(empirical_cms['xgb_clean'])
S_xgb = sensitivity_analysis(V_nominal, a_OTT, C_xgb, t_OTT)

plot_sensitivity(S_xgb, 'Sensitivity $\\partial B_k / \\partial C_{ij}$ (XGBoost Clean)')


In [ ]:
print_top_sensitivities(S_xgb, "Top-10 most sensitive entries S[k,i,j] (XGBoost Clean):")

print("\nKey finding: entries where a_i is large AND t_j >> t_i dominate.")

In [ ]:
# Sensitivity heatmaps: FlowPic Tor CM
C_tor = fix_zero_rows(empirical_cms['flowpic_tor'])
S_tor = sensitivity_analysis(V_nominal, a_OTT, C_tor, t_OTT)

plot_sensitivity(S_tor, 'Sensitivity $\\partial B_k / \\partial C_{ij}$ (FlowPic Tor)')

print_top_sensitivities(S_tor, "\nTop-10 most sensitive entries S[k,i,j] (FlowPic Tor):")


## 6. Capacity Overhead Curves

Sweeping recall from 0.50 to 0.99 under three spillover models (uniform, worst-case, best-case) reveals how spillover direction, not just accuracy, determines both capacity overhead and user-side harm.

- **Uniform:** errors spread equally to all other classes.
- **Worst-case:** all errors directed to Streaming ($t = 15$, highest demand).
- **Best-case:** all errors directed to VoIP ($t = 1$, lowest demand).

In [ ]:
# Capacity overhead + WBD sweep over recall for 3 spillover models. The synthetic matrices come from the archive, keyed by model and recall.
recalls = np.arange(0.50, 1.00, 0.01)

overhead_uniform, wbd_uniform = overhead_wbd_sweep(
    lambda r: data[f'uniform_r{r:.2f}'].copy(), recalls, a_OTT, t_OTT, V_nominal)
overhead_worst, wbd_worst = overhead_wbd_sweep(
    lambda r: data[f'worst_r{r:.2f}'].copy(), recalls, a_OTT, t_OTT, V_nominal)
overhead_best, wbd_best = overhead_wbd_sweep(
    lambda r: data[f'best_r{r:.2f}'].copy(), recalls, a_OTT, t_OTT, V_nominal)

print(f"Sweep complete: {len(recalls)} recall values x 3 spillover models")

In [ ]:
# Plot: Recall vs Capacity Overhead (3 spillover models)
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Plot 1: Capacity Overhead
ax = axes[0]
ax.plot(recalls * 100, overhead_uniform, 'b-', linewidth=2, label='Uniform')
ax.plot(recalls * 100, overhead_worst, 'r-', linewidth=2, label='Worst (all to Streaming)')
ax.plot(recalls * 100, overhead_best, 'g-', linewidth=2, label='Best (all to VoIP)')
for eps_val, ls in [(5, 'dashed'), (10, ':'), (15, '-.')]:
    ax.axhline(y=eps_val, color='gray', linestyle=ls, alpha=0.7, label=f'$\\epsilon$={eps_val}%')
ax.set_xlabel('Per-Class Recall (%)', fontsize=13)
ax.set_ylabel('Capacity Overhead $\\Delta V / V$ (%)', fontsize=13)
ax.set_title('Capacity Overhead vs Recall', fontsize=14)
ax.legend(fontsize=11, loc='upper right')
ax.tick_params(axis='both', labelsize=11)
ax.set_xlim(50, 100)
ax.grid(alpha=0.3)

# Plot 2: WBD
ax = axes[1]
ax.plot(recalls * 100, wbd_uniform, 'b-', linewidth=2, label='Uniform')
ax.plot(recalls * 100, wbd_worst, 'r-', linewidth=2, label='Worst')
ax.plot(recalls * 100, wbd_best, 'g-', linewidth=2, label='Best')
ax.set_xlabel('Per-Class Recall (%)', fontsize=13)
ax.set_ylabel('WBD (AU·Erl)', fontsize=13)
ax.set_title('Weighted Bandwidth Deficit vs Recall', fontsize=14)
ax.legend(fontsize=11, loc='upper right')
ax.tick_params(axis='both', labelsize=11)
ax.set_xlim(50, 100)
ax.grid(alpha=0.3)

# Plot 3: Overhead range comparison
ax = axes[2]
ax.fill_between(recalls * 100, overhead_best, overhead_worst,
                alpha=0.2, color='purple', label='Range (best to worst)')
ax.plot(recalls * 100, overhead_uniform, 'b-', linewidth=2, label='Uniform')
ax.set_xlabel('Per-Class Recall (%)', fontsize=13)
ax.set_ylabel('Capacity Overhead (%)', fontsize=13)
ax.set_title('Overhead Range: Same Recall, Different Spillover', fontsize=14)
ax.legend(fontsize=11, loc='upper right')
ax.tick_params(axis='both', labelsize=11)
ax.set_xlim(50, 100)
ax.grid(alpha=0.3)

plt.subplots_adjust(wspace=0.35)
plt.show()

# Highlight dramatic difference
r_idx_80 = np.argmin(np.abs(recalls - 0.80))
print(f"\nAt recall = 80%:")
print(f"  Worst-case overhead: {overhead_worst[r_idx_80]:.1f}%")
print(f"  Uniform overhead:    {overhead_uniform[r_idx_80]:.1f}%")
print(f"  Best-case overhead:  {overhead_best[r_idx_80]:.1f}%")
print(f"  Worst-case WBD:      {wbd_worst[r_idx_80]:.1f} AU·Erl")
print(f"  Best-case WBD:       {wbd_best[r_idx_80]:.1f} AU·Erl")


## 7. Minimum Recall Thresholds ($r^*_k$)

For each class $k$, the minimum recall $r^*_k$ is the lowest per-class recall at which the capacity overhead stays within a tolerance $\epsilon$. If $r^*_k$ correlates with the class's offered load weight $a_k \cdot t_k$, then high-bandwidth, high-volume classes require stricter classification accuracy (H3).

In [ ]:
# Per-class minimum recall thresholds using uniform spillover
epsilons = [0.05, 0.10, 0.15]
at_k = a_OTT * t_OTT  # Offered load weight per class

print("Per-Class Minimum Recall r*_k (Uniform Spillover, OTT/IPTV)")
print(f"{'Class':>14s}  {'a_k*t_k':>8s}", end="")
for eps in epsilons:
    print(f"  {'eps='+str(int(eps*100))+'%':>8s}", end="")
print()

rstar_table = rstar_by_eps(a_OTT, t_OTT, V_nominal, epsilons)

for i, cls in enumerate(CLASS_ORDER):
    print(f"{cls:>14s}  {at_k[i]:8.1f}", end="")
    for eps in epsilons:
        print(f"  {rstar_table[eps][i]:8.4f}", end="")
    print()

# Also compute global r* for comparison
print(f"\n{'Global r*':>14s}  {'':>8s}", end="")
for eps in epsilons:
    r_global = minimum_recall_search(a_OTT, t_OTT, V_nominal, B_TARGET, eps)
    print(f"  {r_global:8.4f}", end="")
print()

In [ ]:
# H3 test: r*_k vs a_k * t_k scatter with Spearman correlation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax_idx, eps in enumerate(epsilons):
    ax = axes[ax_idx]
    r_stars = rstar_table[eps]

    for i, cls in enumerate(CLASS_ORDER):
        ax.scatter(at_k[i], r_stars[i], s=100, c=CLASS_COLORS[cls],
                   edgecolors='black', linewidths=0.8, zorder=5, label=cls)

    # Spearman correlation
    rho, pval = stats.spearmanr(at_k, r_stars)

    # Linear regression for visual reference
    slope, intercept, r_val, p_val, std_err = stats.linregress(at_k, r_stars)
    x_line = np.linspace(at_k.min() * 0.8, at_k.max() * 1.1, 100)
    ax.plot(x_line, slope * x_line + intercept, color='k', linestyle='dashed', alpha=0.5)

    ax.set_xlabel('$a_k \\cdot t_k$ (AU·Erl)', fontsize=12)
    ax.set_ylabel('$r^*_k$ (minimum recall)', fontsize=12)
    ax.tick_params(axis='both', labelsize=10)
    if np.isnan(rho):
        ax.set_title(f'$\\epsilon$ = {int(eps*100)}%  |  $r^*_k$ constant', fontsize=12)
    else:
        ax.set_title(f'$\\epsilon$ = {int(eps*100)}%  |  $\\rho_s$ = {rho:.3f}, p = {pval:.3f}', fontsize=12)
    if ax_idx == 0:
        ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('H3: Does $r^*_k$ Correlate with $a_k \\cdot t_k$?', fontsize=14)
fig.tight_layout()
plt.show()

# Print test results
print("H3 Spearman Correlation Tests:")
for eps in epsilons:
    rho, pval = stats.spearmanr(at_k, rstar_table[eps])
    print(f"  eps={int(eps*100)}%: rho={rho:.4f}, p={pval:.4f}")


## 8. Hypothesis Evaluation

Three hypotheses are tested:

- **H1:** Low-to-high misclassification ($t_j > t_i$) produces larger per-unit blocking deviations than high-to-low ($t_j < t_i$).
- **H2:** A higher-accuracy but asymmetrically confused CM can produce worse capacity overhead than a lower-accuracy but balanced CM.
- **H3:** The minimum recall threshold $r^*_k$ correlates positively with the class's offered load weight $a_k \cdot t_k$.

In [ ]:
# H1: Low-to-high vs high-to-low per-unit blocking deviation
print("H1: Low-to-high vs High-to-low Misclassification Impact")

# Use XGBoost Clean sensitivity tensor for analytical evidence
print("\n== Evidence from Sensitivity Tensor (XGBoost Clean) ==")
low_to_high_sum = 0.0
high_to_low_sum = 0.0
low_to_high_count = 0
high_to_low_count = 0

for k in range(5):
    for i in range(5):
        for j in range(5):
            if i != j:
                if t_OTT[j] > t_OTT[i]:  # low-to-high
                    low_to_high_sum += abs(S_xgb[k, i, j])
                    low_to_high_count += 1
                elif t_OTT[j] < t_OTT[i]:  # high-to-low
                    high_to_low_sum += abs(S_xgb[k, i, j])
                    high_to_low_count += 1

l2h_mean = low_to_high_sum / max(low_to_high_count, 1)
h2l_mean = high_to_low_sum / max(high_to_low_count, 1)
print(f"  Mean |sensitivity| for low-to-high entries: {l2h_mean:.6f} ({low_to_high_count} entries)")
print(f"  Mean |sensitivity| for high-to-low entries: {h2l_mean:.6f} ({high_to_low_count} entries)")
print(f"  Ratio (low-to-high / high-to-low): {l2h_mean / max(h2l_mean, 1e-12):.2f}x")

# Use synthetic directional CMs for additional evidence
print("\n== Evidence from Synthetic Directional CMs (r=0.80) ==")
C_worst = fix_zero_rows(data['worst_r0.80'])
C_best = fix_zero_rows(data['best_r0.80'])
C_uniform = fix_zero_rows(data['uniform_r0.80'])

for label, C in [('Worst (to Streaming)', C_worst), ('Best (to VoIP)', C_best), ('Uniform', C_uniform)]:
    result = blocking_deviation(V_nominal, a_OTT, C, t_OTT)
    a_hat = result['a_hat']
    try:
        V_prime = capacity_overhead(a_hat, t_OTT, B_TARGET, V_start=V_nominal, V_max=2000)
        oh = (V_prime - V_nominal) / V_nominal * 100
    except ValueError:
        oh = float('nan')
    wbd = compute_wbd(C, a_OTT, t_OTT)
    print(f"  {label:25s}  Overhead: {oh:6.2f}%  WBD: {wbd:6.1f} AU·Erl  Max|dB|: {np.max(np.abs(result['delta_B'])):.6f}")

print("\n== H1 Verdict ==")
if l2h_mean > h2l_mean:
    print("  SUPPORTED: Low-to-high misclassification entries produce larger per-unit")
    print(f"  sensitivity ({l2h_mean:.6f}) than high-to-low ({h2l_mean:.6f}).")
    print("  The worst-case (all errors to Streaming) produces far greater overhead")
    print("  than best-case (all errors to VoIP) at the same recall.")
else:
    print("  NOT SUPPORTED: Evidence does not confirm H1.")

In [ ]:
# H2: Higher accuracy + asymmetric can be worse than lower accuracy + balanced
print("H2: Asymmetric High-Accuracy vs Balanced Low-Accuracy")

# Search empirical CMs for a pair demonstrating this
print("\n== Searching Empirical CM Pairs ==")

for i_name in EMPIRICAL_CM_NAMES:
    for j_name in EMPIRICAL_CM_NAMES:
        if i_name >= j_name:
            continue
        acc_i = compute_accuracy(empirical_cms[i_name])
        acc_j = compute_accuracy(empirical_cms[j_name])
        
        C_i = fix_zero_rows(empirical_cms[i_name])
        C_j = fix_zero_rows(empirical_cms[j_name])
        
        a_hat_i = bridge_equation(C_i, a_OTT)
        a_hat_j = bridge_equation(C_j, a_OTT)
        
        try:
            V_i = capacity_overhead(a_hat_i, t_OTT, B_TARGET, V_start=V_nominal, V_max=2000)
            oh_i = (V_i - V_nominal) / V_nominal * 100
        except ValueError:
            continue
        try:
            V_j = capacity_overhead(a_hat_j, t_OTT, B_TARGET, V_start=V_nominal, V_max=2000)
            oh_j = (V_j - V_nominal) / V_nominal * 100
        except ValueError:
            continue
        
        # Check: higher accuracy but worse overhead
        if acc_i > acc_j and oh_i > oh_j:
            print(f"  FOUND: {CM_LABELS[i_name]} (acc={acc_i:.3f}, oh={oh_i:.1f}%)")
            print(f"       > {CM_LABELS[j_name]} (acc={acc_j:.3f}, oh={oh_j:.1f}%)")
            print(f"  Higher accuracy ({CM_LABELS[i_name]}) produces WORSE overhead!")
        elif acc_j > acc_i and oh_j > oh_i:
            print(f"  FOUND: {CM_LABELS[j_name]} (acc={acc_j:.3f}, oh={oh_j:.1f}%)")
            print(f"       > {CM_LABELS[i_name]} (acc={acc_i:.3f}, oh={oh_i:.1f}%)")
            print(f"  Higher accuracy ({CM_LABELS[j_name]}) produces WORSE overhead!")

# The synthetic pair is shown whatever the empirical search finds.
# CM_A: 90% recall, all errors to Streaming (index 3, the worst direction).
C_A = np.zeros((5, 5))
C_A[3, 3] = 1.0
for i in range(5):
    if i != 3:
        C_A[i, i] = 0.90
        C_A[i, 3] = 0.10

# CM_B: 80% accuracy, errors spread uniformly
C_B = np.full((5, 5), 0.05)
np.fill_diagonal(C_B, 0.80)

acc_A = compute_accuracy(C_A)
acc_B = compute_accuracy(C_B)
a_hat_A = bridge_equation(C_A, a_OTT)
a_hat_B = bridge_equation(C_B, a_OTT)
try:
    V_A = capacity_overhead(a_hat_A, t_OTT, B_TARGET, V_start=V_nominal, V_max=2000)
    oh_A = (V_A - V_nominal) / V_nominal * 100
except ValueError:
    oh_A = float('nan')
try:
    V_B = capacity_overhead(a_hat_B, t_OTT, B_TARGET, V_start=V_nominal, V_max=2000)
    oh_B = (V_B - V_nominal) / V_nominal * 100
except ValueError:
    oh_B = float('nan')

print(f"\n  Synthetic CM_A (asymmetric): acc={acc_A:.3f}, overhead={oh_A:.1f}%")
print(f"  Synthetic CM_B (balanced):   acc={acc_B:.3f}, overhead={oh_B:.1f}%")

print("\n== H2 Verdict ==")
if oh_A > oh_B and acc_A > acc_B:
    print("  SUPPORTED: The higher-accuracy asymmetric CM (90%, errors to Streaming)")
    print(f"  produces {oh_A:.1f}% overhead vs {oh_B:.1f}% for the lower-accuracy balanced CM (80%).")
    print("  Accuracy alone is an insufficient predictor of capacity impact.")
else:
    print("  Result: asymmetric CM overhead={:.1f}%, balanced CM overhead={:.1f}%".format(oh_A, oh_B))
    print("  The direction of errors matters as much as their magnitude.")

In [ ]:
# H3: r*_k correlation with a_k * t_k (summary from Section 7)
print("H3: r*_k Correlation with a_k * t_k")

for eps in epsilons:
    rho, pval = stats.spearmanr(at_k, rstar_table[eps])
    sig = "significant (p < 0.05)" if pval < 0.05 else "not significant (p >= 0.05)"
    print(f"\n  epsilon = {int(eps*100)}%:")
    print(f"    Spearman rho = {rho:.4f}, p-value = {pval:.4f} ({sig})")
    print(f"    r*_k values: {[f'{v:.3f}' for v in rstar_table[eps]]}")
    print(f"    a_k*t_k:     {at_k}")

# Analyze the physical mechanism: r*_k depends on demand gap, not a_k*t_k
print("\n== Physical Interpretation ==")
print("  r*_k per class (eps=5%):")
for i, cls in enumerate(CLASS_ORDER):
    max_gap = max(t_OTT[j] - t_OTT[i] for j in range(len(t_OTT)) if j != i)
    print(f"    {cls:14s}  t_k={t_OTT[i]:2d}  a_k*t_k={at_k[i]:5.0f}  r*_k={rstar_table[0.05][i]:.3f}  max(t_j-t_k)={max_gap:+d}")

print("\n== H3 Verdict ==")
rho_5, pval_5 = stats.spearmanr(at_k, rstar_table[0.05])
print(f"  Spearman rho(a_k*t_k, r*_k) = {rho_5:.4f}")
print("  The simple positive correlation hypothesized by H3 is NOT supported.")
print("  However, a stronger result emerges: r*_k is governed by the maximum")
print("  demand gap max(t_j - t_k) between class k and its spillover targets.")
print("  Classes with LOW t_k (Browsing t=2, VoIP t=1) require the strictest")
print("  recall because their errors spill to high-demand classes (Streaming t=15),")
print("  causing massive demand inflation. Classes with HIGH t_k (Streaming, FT)")
print("  can tolerate low recall because their errors spill DOWN, reducing demand.")
print("  This is fully consistent with H1 and reinforces the thesis argument that")
print("  spillover DIRECTION, not aggregate accuracy, governs capacity impact.")

The negative rank correlation above is the $K = 5$ OTT/IPTV scenario. On the $K = 23$ CESNET anchor (`scripts/cesnet/cesnet_highk.py`) the bandwidth-gap correlation is $+0.610$ ($p = 0.003$), which is the result Chapter 5 reports. A `nan` Spearman value appears wherever every $r^*_k$ sits at the $0.50$ floor, so the statistic is undefined there.

## 9. 5G Slicing Scenario (3-class)

The 5G scenario uses three slice types (eMBB, mMTC, URLLC) with synthetic 3x3 CMs. This provides a cross-scenario comparison and tests whether the OTT/IPTV findings generalize to a different demand structure.

- **Worst-case:** all errors directed to eMBB ($t = 10$, highest demand).
- **Best-case:** all errors directed to mMTC ($t = 1$, lowest demand).

In [ ]:
# 5G: synthetic CMs from src/analytical/scenarios.make_5g_cm, the same code path that writes the 5G keys of analytical_results.npz. "worst" sends all error mass to eMBB (t=10), "best" to mMTC (t=1).

# Find 5G nominal capacity
V_5g_nominal = capacity_overhead(a_5G, t_5G, B_TARGET, V_start=1, V_max=500)
print(f"5G Nominal capacity V* = {V_5g_nominal} AUs")
_, B_5g_nom = kaufman_roberts(V_5g_nominal, a_5G, t_5G)
print(f"Blocking at V*: {[f'{b:.6f}' for b in B_5g_nom]}")

# Blocking deviation for sample CMs
print("\n5G Blocking Deviation (r=0.80):")
print("=" * 60)
for spill_type in ['uniform', 'worst', 'best']:
    C = make_5g_cm(0.80, spill_type)
    result = blocking_deviation(V_5g_nominal, a_5G, C, t_5G)
    a_hat = result['a_hat']
    try:
        V_prime = capacity_overhead(a_hat, t_5G, B_TARGET, V_start=V_5g_nominal, V_max=2000)
        oh = (V_prime - V_5g_nominal) / V_5g_nominal * 100
    except ValueError:
        oh = float('nan')
    wbd = compute_wbd(C, a_5G, t_5G)
    print(f"  {spill_type:8s}  Overhead: {oh:6.2f}%  WBD: {wbd:6.1f} AU·Erl")

In [ ]:
# 5G: Sensitivity heatmaps (uniform CM at r=0.90)
C_5g_uniform = make_5g_cm(0.90, 'uniform')
S_5g = sensitivity_analysis(V_5g_nominal, a_5G, C_5g_uniform, t_5G)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle('5G Sensitivity $\\partial B_k / \\partial C_{ij}$ (Uniform, r=0.90)', fontsize=14)

vmax_5g = np.max(np.abs(S_5g))
for k in range(3):
    ax = axes[k]
    im = ax.imshow(S_5g[k], cmap='RdBu_r', vmin=-vmax_5g, vmax=vmax_5g, aspect='equal')
    ax.set_title(f'$B_{{{CLASSES_5G[k]}}}$', fontsize=12)
    ax.set_xticks(range(3))
    ax.set_xticklabels(CLASSES_5G, rotation=45, ha='right', fontsize=10)
    ax.set_yticks(range(3))
    ax.set_yticklabels(CLASSES_5G, fontsize=10)
    if k == 0:
        ax.set_ylabel('True class $i$', fontsize=11)
    ax.set_xlabel('Predicted class $j$', fontsize=11)

    for i in range(3):
        for j in range(3):
            val = S_5g[k, i, j]
            color = 'white' if abs(val) > vmax_5g * 0.6 else 'black'
            ax.text(j, i, f'{val:.4f}', ha='center', va='center', fontsize=9, color=color)

plt.subplots_adjust(right=0.82, wspace=0.4, top=0.88)
cbar_ax = fig.add_axes([0.85, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='$\\partial B_k / \\partial C_{ij}$')

plt.show()


In [ ]:
# 5G: Capacity overhead curves
recalls_5g = np.arange(0.50, 1.00, 0.01)

oh_5g_uniform, wbd_5g_uniform = overhead_wbd_sweep(
    lambda r: make_5g_cm(r, 'uniform'), recalls_5g, a_5G, t_5G, V_5g_nominal)
oh_5g_worst, wbd_5g_worst = overhead_wbd_sweep(
    lambda r: make_5g_cm(r, 'worst'), recalls_5g, a_5G, t_5G, V_5g_nominal)
oh_5g_best, wbd_5g_best = overhead_wbd_sweep(
    lambda r: make_5g_cm(r, 'best'), recalls_5g, a_5G, t_5G, V_5g_nominal)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.plot(recalls_5g * 100, oh_5g_uniform, 'b-', linewidth=2, label='Uniform')
ax.plot(recalls_5g * 100, oh_5g_worst, 'r-', linewidth=2, label='Worst (to eMBB)')
ax.plot(recalls_5g * 100, oh_5g_best, 'g-', linewidth=2, label='Best (to mMTC)')
for eps_val, ls in [(5, 'dashed'), (10, ':')]:
    ax.axhline(y=eps_val, color='gray', linestyle=ls, alpha=0.7, label=f'$\\epsilon$={eps_val}%')
ax.set_xlabel('Per-Class Recall (%)', fontsize=12)
ax.set_ylabel('Capacity Overhead (%)', fontsize=12)
ax.set_title('5G: Capacity Overhead vs Recall', fontsize=14)
ax.legend(fontsize=10)
ax.tick_params(axis='both', labelsize=10)
ax.set_xlim(50, 100)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(recalls_5g * 100, wbd_5g_uniform, 'b-', linewidth=2, label='Uniform')
ax.plot(recalls_5g * 100, wbd_5g_worst, 'r-', linewidth=2, label='Worst')
ax.plot(recalls_5g * 100, wbd_5g_best, 'g-', linewidth=2, label='Best')
ax.set_xlabel('Per-Class Recall (%)', fontsize=12)
ax.set_ylabel('WBD (AU·Erl)', fontsize=12)
ax.set_title('5G: WBD vs Recall', fontsize=14)
ax.legend(fontsize=10)
ax.tick_params(axis='both', labelsize=10)
ax.set_xlim(50, 100)
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()


In [ ]:
# 5G: Minimum recall thresholds + cross-scenario comparison
at_5g = a_5G * t_5G

print("5G Per-Class Minimum Recall r*_k (Uniform Spillover)")
rstar_5g = rstar_by_eps(a_5G, t_5G, V_5g_nominal, epsilons)
for eps, r_stars in rstar_5g.items():
    print(f"\n  eps={int(eps*100)}%: {[f'{v:.3f}' for v in r_stars]}")
    print(f"  a_k*t_k: {at_5g}")
    rho, pval = stats.spearmanr(at_5g, r_stars)
    print(f"  Spearman rho={rho:.4f}, p={pval:.4f}")

# Cross-scenario comparison table
print("\n\nCross-Scenario Comparison (eps=5%, Uniform Spillover)")
print(f"{'Metric':>30s}  {'OTT/IPTV':>12s}  {'5G Slicing':>12s}")
print(f"{'Classes':>30s}  {5:>12d}  {3:>12d}")
print(f"{'V_nominal':>30s}  {V_nominal:>12d}  {V_5g_nominal:>12d}")
print(f"{'Total AU·Erl':>30s}  {np.sum(a_OTT * t_OTT):>12.1f}  {np.sum(a_5G * t_5G):>12.1f}")
print(f"{'Max t_k / Min t_k':>30s}  {max(t_OTT)/min(t_OTT):>12.0f}  {max(t_5G)/min(t_5G):>12.0f}")

# Overhead at r=0.80 uniform
r80_idx = np.argmin(np.abs(recalls - 0.80))
r80_5g_idx = np.argmin(np.abs(recalls_5g - 0.80))
print(f"{'Overhead at r=0.80 (uniform)':>30s}  {overhead_uniform[r80_idx]:>11.1f}%  {oh_5g_uniform[r80_5g_idx]:>11.1f}%")
print(f"{'WBD at r=0.80 (uniform)':>30s}  {wbd_uniform[r80_idx]:>12.1f}  {wbd_5g_uniform[r80_5g_idx]:>12.1f}")

# Global r* comparison
r_ott_global = minimum_recall_search(a_OTT, t_OTT, V_nominal, B_TARGET, 0.05)
r_5g_global = minimum_recall_search(a_5G, t_5G, V_5g_nominal, B_TARGET, 0.05)
print(f"{'Global r* (eps=5%)':>30s}  {r_ott_global:>12.4f}  {r_5g_global:>12.4f}")

## 10. Summary

Collect the analytical results and print the summary tables used in Chapter 5.

## Summary: the dual-impact reading of classification errors

1. **Baseline.** Both scenarios are dimensioned so that baseline blocking meets the
   1 percent grade-of-service target. The nominal capacities are printed above.
2. **Bridge equation.** Classification errors redistribute offered load across
   classes. Total AU-Erlang demand may rise (low-to-high dominant) or fall
   (high-to-low dominant), depending on the structure of the confusion matrix.
3. **Capacity overhead, the system cost.** Worst-case spillover, with every error
   sent to the highest-demand class, produces far more overhead than uniform or
   best-case spillover at the same recall. Zero overhead does not mean zero harm:
   it can mean high-to-low errors that lower apparent demand while degrading
   per-flow quality of service.
4. **Weighted bandwidth deficit, the user cost.** The WBD measures the harm to
   individual flows when high-demand traffic is classified as low-demand, which
   the system-level overhead figure misses.
5. **Sensitivity.** The largest entries are those where the offered load $a_i$ is
   large and $t_j \gg t_i$, the analytical evidence for H1. The ratio between the
   two directions is computed in the H1 cell above.
6. **Minimum recall.** Per-class thresholds $r^*_k$ are governed by the maximum
   demand gap $\max_j(t_j - t_k)$ rather than by $a_k t_k$. Low-demand classes
   need the strictest recall because their errors inflate demand most.
7. **Cross-scenario.** OTT/IPTV and 5G show the same qualitative patterns over
   two different demand structures.
